**Author**: Felipe Matheus  
**Start Date**: 13/08/2026  
**End Date**: 18/08/2026  
**Purpose**: Explore the new developed features IRI and GBEI.


# 1. Setup

In [ ]:
import logging
import os
import sys

import numpy as np
import pandas as pd
import pickle
from pathlib import Path

module_path = os.path.abspath(os.path.join('../..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from src.processing.Processing import Processing
from src.feature_engineering.FeatureEngineering import FeatureEngineering
from src.modeling.Modeling import Modeling
from src.metrics.Evaluation import Evaluation
from src.modeling.Experiments import ExperimentConfig, ExperimentRunner

from config.Variables import Variables
from config.bags_compositions import *
from config.elements_coefficients import *

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
)

%load_ext autoreload
%autoreload 2

varv = Variables()
proc = Processing()
feng = FeatureEngineering()
modl = Modeling()
evla = Evaluation()
runr = ExperimentRunner(modl, evla, models_root=varv.PATHS.models)

c:\Users\fmfoa\Projects\uncertainty-aware-predictors\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Download raw data

In [2]:
SCHEMA_DATE = "170826"

FILE_NAME_SCHEMA_DATA = "schema_annealing_essays_{}.csv".format(SCHEMA_DATE)

df_raw = pd.read_csv(os.path.join(varv.PATHS.data_raw, FILE_NAME_SCHEMA_DATA))
FEATURES_TO_SELECT = [
    "experiment_id",
    "material",
    "purity",
    "initial_diameter",
    "temperature",
    "time",
    "iacs",
    "tensile_strength",
    "elongation",
    "iacs_final",
    "tensile_strength_final",
    "elongation_final",
]

In [3]:
df_nnan = df_raw.dropna(how="all")
df_feat = df_nnan[FEATURES_TO_SELECT]

In [4]:
df_nnan.columns

Index(['experiment_id', 'material', 'purity', 'initial_diameter',
       'temperature_C', 'temperature', 'time', 'grain_number',
       'arithmetic_mean_area', 'equivalent_circle_diameter', 'desnity_of_HAGB',
       'EBSD map dimensions', 'lin-resistance', 'iacs', 'tensile_strength',
       'elongation', 'grain_number_final', 'arithmetic_mean_area_final',
       'equivalent_circle_diameter_final', 'Density_of_HAGB_final',
       'EBSD_map_dimensions (um x um)', 'lin-resistance_final', 'iacs_final',
       'tensile_strength_final', 'elongation_final', 'previous_experiment_id',
       'test', 'sample-code', 'MBC-predict', 'Comments'],
      dtype='object')

In [5]:
df_feat.material.unique()

array(['C14500', 'Cu-ETP', 'Cu-T2', 'LT06',
       'lab-cast_Trial-8_1BSac2_Cu-99.98', 'lab-cast_S1_T-10',
       'lab-cast_S2_T-11', 'lab-cast_S4_T14', 'lab-cast_S9_T-20'],
      dtype=object)

In [15]:
ALL_SAMPLES

{'1A sac1': {'Zn': {'val': 0.001, 'sd': 0.0, 'rsd': 0.0, 'below_limit': True},
  'Pb': {'val': 0.0003, 'sd': 0.0, 'rsd': 0.0, 'below_limit': True},
  'Sn': {'val': 0.0002, 'sd': 0.0, 'rsd': 0.0, 'below_limit': True},
  'P': {'val': 0.0002, 'sd': 0.0, 'rsd': 0.0, 'below_limit': True},
  'Mn': {'val': 0.0004, 'sd': 0.0, 'rsd': 0.0, 'below_limit': True},
  'Fe': {'val': 0.0011, 'sd': 0.00022, 'rsd': 20.5, 'below_limit': False},
  'Ni': {'val': 0.0002, 'sd': 0.0, 'rsd': 0.0, 'below_limit': True},
  'Si': {'val': 0.001, 'sd': 9e-05, 'rsd': 9.1, 'below_limit': False},
  'Mg': {'val': 0.00026, 'sd': 1e-05, 'rsd': 2.0, 'below_limit': False},
  'Cr': {'val': 0.0003, 'sd': 0.0, 'rsd': 0.0, 'below_limit': True},
  'As': {'val': 0.00044, 'sd': 5e-05, 'rsd': 11.7, 'below_limit': False},
  'Sb': {'val': 0.0033, 'sd': 0.0002, 'rsd': 5.9, 'below_limit': False},
  'Cd': {'val': 0.0003, 'sd': 0.0, 'rsd': 0.0, 'below_limit': True},
  'Bi': {'val': 0.0005, 'sd': 0.0, 'rsd': 0.0, 'below_limit': True},
  'A

In [ ]:
from src.feature_engineering.IRIHelper import IRIHelper
from src.feature_engineering.GBEIHelper import GBEIHelper
from config.bags_compositions import ALL_SAMPLES
from config.elements_coefficients import RESISTIVITY_FACTORS, ENRICHMENT_FACTORS

iri_df  = IRIHelper(RESISTIVITY_FACTORS).compute(ALL_SAMPLES)
gbei_df = GBEIHelper(ENRICHMENT_FACTORS).compute(ALL_SAMPLES)

In [8]:
iri_df

,Pb (wt%),Te (wt%),regime,Te above threshold,Pb vol fraction,Te vol fraction,IRI (nΩ·m),ρ_matrix (nΩ·m),ρ_ss (nΩ·m),Δρ_Pb (nΩ·m),Δρ_Te (nΩ·m),ρ_total (nΩ·m)
1A sac1,0.00015,0.00170,solid-solution (IRI),False,0.000000,0.000000,0.449385,17.229385,17.288535,0.001350,0.057800,17.288535
1A sac2,0.00015,0.00190,solid-solution (IRI),False,0.000000,0.000000,0.449385,17.229385,17.295335,0.001350,0.064600,17.295335
1B sac1,0.00015,0.00200,solid-solution (IRI),False,0.000000,0.000000,0.867175,17.647175,17.716525,0.001350,0.068000,17.716525
1B sac2,0.00015,0.00200,solid-solution (IRI),False,0.000000,0.000000,0.517175,17.297175,17.366525,0.001350,0.068000,17.366525
1B sac3,0.01000,0.00210,solid-solution (IRI),False,0.000000,0.000000,0.513685,17.293685,17.455085,0.090000,0.071400,17.455085
1B sac4,0.00530,0.00290,solid-solution (IRI),False,0.000000,0.000000,1.173155,17.953155,18.099455,0.047700,0.098600,18.099455
1B sac5,0.00390,0.00200,solid-solution (IRI),False,0.000000,0.000000,0.595135,17.375135,17.478235,0.035100,0.068000,17.478235
1B sac6,0.00015,0.00220,solid-solution (IRI),False,0.000000,0.000000,1.000575,17.780575,17.856725,0.001350,0.074800,17.856725
2N sample,0.12400,0.00220,solid-solution (IRI),False,0.000000,0.000000,1.545235,18.325235,19.516035,1.116000,0.074800,19.516035
2N sac1,0.03320,0.00210,solid-solution (IRI),False,0.000000,0.000000,0.633215,17.413215,17.783415,0.298800,0.071400,17.783415


In [10]:
gbei_df.T

,1A sac1,1A sac2,1B sac1,1B sac2,1B sac3,1B sac4,1B sac5,1B sac6,2N sample,2N sac1,2N sac2,enchantillon_1,enchantillon_2,enchantillon_3,enchantillon_4,Cu-T2,Cu-Chine,CGR-ETP,Cu-ETP-LT4,C14500
GBEI,0.004611,0.004788,0.014312,0.007281,0.016767,0.024596,0.011322,0.018851,0.149803,0.04143,0.296045,0.346285,0.33939,0.353471,0.271706,0.002674,0.045731,0.00104,0.001038,0.4898


# Converting

In [13]:
df_feat.tail(20)

,experiment_id,material,purity,initial_diameter,temperature,time,iacs,tensile_strength,elongation,iacs_final,tensile_strength_final,elongation_final
28,ANE_SN013_CD1_RS1_WP1_202606161745,Cu-ETP,99.9000,1.2,523.0,90.0,101.220,397.09,2.36%,104.520,259.140000,42.28%
30,ANE_SN014_CD1_RS1_WP1_202606021918,LT06,99.9800,1.2,573.0,30.0,102.700,335.21,1.61%,103.295,247.170000,45.97%
31,ANE_SN014_CD1_RS1_WP1_202606021848,lab-cast_Trial-8_1BSac2_Cu-99.98,99.9800,1.2,573.0,60.0,102.700,335.21,1.61%,100.785,245.000000,50.67%
32,ANE_SN014_CD1_RS1_WP1_202606021818,lab-cast_Trial-8_1BSac2_Cu-99.98,99.9800,1.2,573.0,90.0,102.700,335.21,1.61%,103.935,248.620000,46.64%
33,ANE_SN014_CD1_RS1_WP1_202606041150,lab-cast_Trial-8_1BSac2_Cu-99.98,99.9800,1.2,523.0,30.0,102.700,335.21,1.61%,102.235,245.680000,51.01%
34,ANE_SN014_CD1_RS1_WP1_202606041120,lab-cast_Trial-8_1BSac2_Cu-99.98,99.9800,1.2,523.0,60.0,102.700,335.21,1.61%,103.200,246.910000,49.42%
35,ANE_SN014_CD1_RS1_WP1_202606041050,LT06,99.9800,1.2,523.0,90.0,102.700,335.21,1.61%,103.560,245.850000,44.78%
37,ANE_SN015_CD1_RS1_WP1_202607081730,lab-cast_S1_T-10,99.9740,2.0,673.0,30.0,99.650,354.83,5.72%,102.675,246.085711,52.80%
38,ANE_SN015_CD1_RS1_WP1_202607291830,lab-cast_S1_T-10,99.9740,0.8,723.0,60.0,97.565,347.70,3.56%,98.500,249.510000,47.13%
39,ANE_SN015_CD1_RS1_WP1_202607301300,lab-cast_S1_T-10,99.9740,0.8,823.0,0.5,97.565,347.70,3.56%,99.020,301.290000,5.67%


In [11]:
df_feat.material.unique()

array(['C14500', 'Cu-ETP', 'Cu-T2', 'LT06',
       'lab-cast_Trial-8_1BSac2_Cu-99.98', 'lab-cast_S1_T-10',
       'lab-cast_S2_T-11', 'lab-cast_S4_T14', 'lab-cast_S9_T-20'],
      dtype=object)

In [14]:
df_feat

,experiment_id,material,purity,initial_diameter,temperature,time,iacs,tensile_strength,elongation,iacs_final,tensile_strength_final,elongation_final
0,ANE_SN010_CD1_RS1_WP1_202602171500-1,C14500,99.9000,1.900000,623.0,30.0,88.260,NaN,NaN,88.350000,NaN,NaN
1,ANE_SN010_CD1_RS1_WP1_202602171430-1,C14500,99.9000,1.900000,623.0,60.0,88.260,NaN,NaN,88.655000,NaN,NaN
2,ANE_SN010_CD1_RS1_WP1_202602171400-1,C14500,99.9000,1.900000,623.0,90.0,88.260,NaN,NaN,88.800000,NaN,NaN
4,ANE_SN011_CD1_RS1_WP1_202602181507-1,Cu-ETP,99.9000,2.160000,573.0,30.0,99.825,391.690000,4.90%,102.340000,249.783315,48.60%
5,ANE_SN011_CD1_RS1_WP1_202603301900-1,Cu-ETP,99.9000,2.182333,573.0,30.0,99.390,387.595254,6.02%,102.230000,255.008013,56.21%
7,ANE_SN011_CD1_RS1_WP1_202603311135-1,Cu-ETP,99.9000,2.182333,588.0,30.0,99.390,387.595254,6.02%,101.346667,NaN,NaN
8,ANE_SN011_CD1_RS1_WP1_202604031750-1,Cu-ETP,99.9000,2.191700,573.0,30.0,99.230,404.662322,3.53%,101.355000,NaN,NaN
9,ANE_SN011_CD1_RS1_WP1_202604031750-2,Cu-ETP,99.9000,2.196300,573.0,30.0,98.720,411.951845,3.27%,101.480000,NaN,NaN
11,NaN,Cu-T2,99.9000,1.200000,523.0,30.0,97.710,431.580000,0.94%,100.290000,247.080000,20.15%
12,NaN,Cu-T2,99.9000,1.200000,523.0,60.0,97.710,431.580000,0.94%,100.420000,242.210000,22.04%


# IRI 

In [ ]:
C14500 = {
    "Cu": [99.2, 99.596],
    "Te": [0.4, 0.7],
    "P": [0.0040, 0.012]
}

In [16]:
avgs = [np.average(val) for val in C14500.values()]
avgs_corrected = [avg / sum(avgs) for avg in avgs]

In [21]:
[a*100 for a in avgs_corrected]

[np.float64(99.44175437192365),
 np.float64(0.550242106526872),
 np.float64(0.008003521549481772)]

In [23]:
100 - (0.55+99.44)

0.010000000000005116

In [15]:
sum(avgs)

np.float64(99.95599999999999)

In [8]:

# ============================================================
# 1. Build one DataFrame per sample
# ============================================================

dfs = {}

for sample_name, elements in ALL_SAMPLES.items():

    rows = []

    for element, data in elements.items():
        rows.append({
            "element": element,
            "val_%": data["val"],
            "sd": data["sd"],
            "rsd_%": data["rsd"],
            "below_limit": data["below_limit"],
        })

    df = pd.DataFrame(rows).set_index("element")

    # If below detection limit, use half of reported value
    df["concentration (wt%)"] = df.apply(
        lambda row: (
            row["val_%"] / 2
            if row["below_limit"]
            else row["val_%"]
        ),
        axis=1
    )

    dfs[sample_name] = df


# ============================================================
# 2. Constants
# ============================================================

rho_Cu_pure = 16.78       # nΩ·m
rho_Pb_bulk = 208.0       # nΩ·m

density_Pb = 11.34        # g/cm³
density_Cu = 8.96         # g/cm³

PB_THRESHOLD = 0.14       # wt%
                          # Above this: Pb treated as second phase


# ============================================================
# 3. Functions
# ============================================================

def maxwell_garnett(
    rho_matrix: float,
    rho_inclusion: float,
    f_vol: float
) -> float:
    """
    Maxwell-Garnett effective resistivity for Pb particles
    dispersed in the Cu matrix.
    """

    num = (
        rho_inclusion
        + 2 * rho_matrix
        + 2 * f_vol * (rho_inclusion - rho_matrix)
    )

    den = (
        rho_inclusion
        + 2 * rho_matrix
        - f_vol * (rho_inclusion - rho_matrix)
    )

    return rho_matrix * (num / den)


def wt_to_vol_fraction(w_Pb_pct: float) -> float:
    """
    Convert Pb concentration from wt% to volume fraction.
    """

    w = w_Pb_pct / 100.0

    return (
        (w / density_Pb)
        /
        (
            (w / density_Pb)
            + ((1 - w) / density_Cu)
        )
    )


# ============================================================
# 4. Calculate IRI + Pb contribution
# ============================================================

results = {}

for sample_name, df in dfs.items():

    iri = 0.0
    pb_concentration_wt = 0.0

    # --------------------------------------------------------
    # Calculate IRI for all elements except Pb
    # --------------------------------------------------------

    for element, row in df.iterrows():

        if element == "Pb":
            pb_concentration_wt = row["concentration (wt%)"]
            continue

        resistivity_factor = RESISTIVITY_FACTORS.get(
            element,
            0.0
        )

        iri += (
            row["concentration (wt%)"]
            * resistivity_factor
        )

    # --------------------------------------------------------
    # Cu matrix resistivity
    # --------------------------------------------------------

    rho_matrix = rho_Cu_pure + iri


    # --------------------------------------------------------
    # Pb routing
    # --------------------------------------------------------

    if pb_concentration_wt > PB_THRESHOLD:

        # Pb above threshold:
        # treat Pb as second-phase particles

        f_vol = wt_to_vol_fraction(
            pb_concentration_wt
        )

        rho_total = maxwell_garnett(
            rho_matrix,
            rho_Pb_bulk,
            f_vol
        )

        delta_rho_Pb = rho_total - rho_matrix

        pb_regime = "second-phase (MG)"

    else:

        # Pb below threshold:
        # treat Pb as dissolved solid-solution impurity

        f_vol = 0.0

        pb_factor = RESISTIVITY_FACTORS.get(
            "Pb",
            0.0
        )

        delta_rho_Pb = (
            pb_concentration_wt
            * pb_factor
        )

        rho_total = rho_matrix + delta_rho_Pb

        pb_regime = "solid-solution (IRI)"


    # --------------------------------------------------------
    # Store results
    # --------------------------------------------------------

    results[sample_name] = {

        "Pb (wt%)":
            pb_concentration_wt,

        "Pb regime":
            pb_regime,

        "Pb vol fraction":
            f_vol,

        "IRI (nΩ·m)":
            iri,

        "ρ_matrix (nΩ·m)":
            rho_matrix,

        "Δρ_Pb (nΩ·m)":
            delta_rho_Pb,

        "ρ_total (nΩ·m)":
            rho_total,
    }


# ============================================================
# 5. Final results DataFrame
# ============================================================

results_df = pd.DataFrame.from_dict(
    results,
    orient="index"
)

print(results_df)

                Pb (wt%)             Pb regime  Pb vol fraction  IRI (nΩ·m)  \
1A sac1          0.00015  solid-solution (IRI)         0.000000    0.507185   
1A sac2          0.00015  solid-solution (IRI)         0.000000    0.513985   
1B sac1          0.00015  solid-solution (IRI)         0.000000    0.935175   
1B sac2          0.00015  solid-solution (IRI)         0.000000    0.585175   
1B sac3          0.01000  solid-solution (IRI)         0.000000    0.585085   
1B sac4          0.00530  solid-solution (IRI)         0.000000    1.271755   
1B sac5          0.00390  solid-solution (IRI)         0.000000    0.663135   
1B sac6          0.00015  solid-solution (IRI)         0.000000    1.075375   
2N sample        0.12400  solid-solution (IRI)         0.000000    1.620035   
2N sac1          0.03320  solid-solution (IRI)         0.000000    0.704615   
2N sac2          0.30200     second-phase (MG)         0.002388    0.704965   
enchantillon_1   0.35400     second-phase (MG)      

In [9]:
results_df

,Pb (wt%),Pb regime,Pb vol fraction,IRI (nΩ·m),ρ_matrix (nΩ·m),Δρ_Pb (nΩ·m),ρ_total (nΩ·m)
1A sac1,0.00015,solid-solution (IRI),0.000000,0.507185,17.287185,0.001350,17.288535
1A sac2,0.00015,solid-solution (IRI),0.000000,0.513985,17.293985,0.001350,17.295335
1B sac1,0.00015,solid-solution (IRI),0.000000,0.935175,17.715175,0.001350,17.716525
1B sac2,0.00015,solid-solution (IRI),0.000000,0.585175,17.365175,0.001350,17.366525
1B sac3,0.01000,solid-solution (IRI),0.000000,0.585085,17.365085,0.090000,17.455085
1B sac4,0.00530,solid-solution (IRI),0.000000,1.271755,18.051755,0.047700,18.099455
1B sac5,0.00390,solid-solution (IRI),0.000000,0.663135,17.443135,0.035100,17.478235
1B sac6,0.00015,solid-solution (IRI),0.000000,1.075375,17.855375,0.001350,17.856725
2N sample,0.12400,solid-solution (IRI),0.000000,1.620035,18.400035,1.116000,19.516035
2N sac1,0.03320,solid-solution (IRI),0.000000,0.704615,17.484615,0.298800,17.783415


In [ ]:
#TODO receive 